# 变系数 NLSE 光孤子不确定性 —— 一键复现

本 Notebook 把复现包的计算按顺序串起来，**点「运行全部 / Run All」即可到底**。

- `code/`：全部计算脚本（原样保留，未改动）
- `results/`：数值结果、样本、表格宏与审计记录（本包已附原始产物）
- `figures/`：4 张投稿级插图（PDF + 600 dpi PNG）

**运行模式**：下方 `MODE` 单元格里，
 - `"reuse"`（默认）：直接复用 `results/` 中已有数据，秒级重建报表、表格和全部插图，适合查看与核对；
 - `"full"`：从头重算全部数值（含管理链路，约 24 分钟；物理光纤算例另需数分钟），适合真正复算。

依赖：`numpy scipy sympy matplotlib`。`requirements.txt` 在 `code/` 下。

In [ ]:
# ===== MODE: 'reuse' 复用已有结果（快） | 'full' 从头完整重算（慢） =====
MODE = 'reuse'

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

# 定位根目录：优先 notebook 所在目录；若在子目录运行则向上找到含 code/ 的层级
def find_root(start):
    p = Path(start).resolve()
    for q in [p] + list(p.parents):
        if (q / 'code' / 'study.py').is_file():
            return q
    return p

try:
    HERE = Path.cwd()
except Exception:
    HERE = Path('.')
ROOT = find_root(HERE)
CODE = ROOT / 'code'
RES  = ROOT / 'results'
FIG  = ROOT / 'figures'
os.chdir(ROOT)
sys.path.insert(0, str(CODE))

print('ROOT =', ROOT)
print('python =', sys.version.split()[0])
for mod in ('numpy', 'scipy', 'sympy', 'matplotlib'):
    try:
        m = __import__(mod)
        print(f'  {mod:11s} {getattr(m, "__version__", "?")}')
    except Exception as e:
        print(f'  {mod:11s} MISSING ({e})')
assert MODE in ('reuse', 'full'), "MODE must be 'reuse' or 'full'"
print('MODE =', MODE)

In [ ]:
# 依赖自举：缺包时按 requirements 安装；网络受限环境可跳过（已装则无操作）
import importlib
missing = [m for m in ('numpy','scipy','sympy','matplotlib')
           if importlib.util.find_spec(m) is None]
if missing:
    print('installing:', missing)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '-r', str(CODE/'requirements.txt')])
    print('pip exit =', r.returncode)
else:
    print('all required packages present; skip install')

## 1. 符号审计（SymPy 独立核对 19 项恒等式）

In [ ]:
# 'full' 模式重算符号审计；'reuse' 直接读取已存结果
if MODE == 'full':
    r = subprocess.run([sys.executable, str(CODE/'symbolic_audit.py')])
    assert r.returncode == 0, 'symbolic_audit.py failed'
d = json.loads((RES/'symbolic_audit.json').read_text(encoding='utf-8'))
print('symbolic checks passed =', d['passed'], '| sympy', d['sympy'])

## 2. 管理链路主研究与样本（最耗时一步）

`full` 模式重跑 `study.py`（约 24 分钟），生成 `study.json`、32 个 `samples_*.npz` 及 4 张中间图；`reuse` 模式直接采用包内已有产物。

In [ ]:
if MODE == 'full':
    t0 = time.time()
    r = subprocess.run([sys.executable, str(CODE/'study.py'),
                        '--power', '8', '--replicates', '4'])
    print('study wall = %.1f s, exit = %d' % (time.time()-t0, r.returncode))
    assert r.returncode == 0, 'study.py failed'
s = json.loads((RES/'study.json').read_text(encoding='utf-8'))
print('records =', len(s['records']), '| best control =', s['design']['best']['control'],
      '| reported wall = %.1f s' % s.get('wall_seconds', 0))

## 3. 光影风险网格 `figures_relief.py`

In [ ]:
if MODE == 'full':
    r = subprocess.run([sys.executable, str(CODE/'figures_relief.py')])
    assert r.returncode == 0, 'figures_relief.py failed'
print('relief_data.npz exists =', (RES/'relief_data.npz').is_file())

## 4. 扩展分析 `extensions.py`（净增益 / 重分布求积）

In [ ]:
if MODE == 'full':
    r = subprocess.run([sys.executable, str(CODE/'extensions.py')])
    assert r.returncode == 0, 'extensions.py failed'
e = json.loads((RES/'extensions.json').read_text(encoding='utf-8'))
print('net gain change = %.4f | redistribution change = %.4f'
      % (e['net_gain_change'], e['redistribution_change']))

## 5. 光纤物理算例 `fiber_case.py`（512 样本）

In [ ]:
if MODE == 'full':
    t0 = time.time()
    r = subprocess.run([sys.executable, str(CODE/'fiber_case.py')])
    print('fiber wall = %.1f s, exit = %d' % (time.time()-t0, r.returncode))
    assert r.returncode == 0, 'fiber_case.py failed'
f = json.loads((RES/'fiber.json').read_text(encoding='utf-8'))
print('fiber keys =', sorted(f.keys()))

## 6. 报表数值与 LaTeX 表格宏 `report_results.py`

In [ ]:
# 论文源文件已随投稿材料移除，报表改用 results/ 中已生成的宏与表格
sm = json.loads((RES/'summary.json').read_text(encoding='utf-8'))
print('paired improvement @0.1 =', sm['paired_improvement_at_0.1'])
for t in ('table_convergence.tex','table_risk.tex','table_shares.tex','macros.tex'):
    p=RES/t
    print(' ', t, p.stat().st_size, 'bytes')
    assert p.is_file(), 'missing '+t


## 7. 投稿级插图 `publication_figures.py`（4 图，PDF + 600 dpi PNG）

In [ ]:
r = subprocess.run([sys.executable, str(CODE/'publication_figures.py')])
assert r.returncode == 0, 'publication_figures.py failed'
for p in sorted(FIG.glob('*')):
    print(' ', p.name, '%.0f KB' % (p.stat().st_size/1024))

## 8. 引文在线核验 `audit_crossref.py`（需联网，可跳过）

该步访问 Crossref 做 DOI 核验。无网络时会自动跳过并沿用包内归档的 `crossref_audit.json`，不影响其余计算。

In [ ]:
# 引文核验源文件已移除，直接复用 results/ 中归档的 Crossref 审计结果
c = json.loads((RES/'crossref_audit.json').read_text(encoding='utf-8'))
print('crossref verified =', c.get('verified'), '/', c.get('checked'))


## 9. 末尾核对

汇总关键证明文件是否齐全，给出 PASS / FAIL 清单。

In [ ]:
checks = {}
need_res = ['study.json','summary.json','symbolic_audit.json','extensions.json',
            'fiber.json','fiber.npz','relief_data.npz','crossref_audit.json',
            'table_convergence.tex','table_risk.tex','table_shares.tex']
need_fig = ['managed_energy_waveform','quadratic_risk','schedule_tradeoffs','dimensioned_fiber']
for n in need_res:
    checks['results/'+n] = (RES/n).is_file()
for n in need_fig:
    checks['figures/'+n+'.pdf'] = (FIG/(n+'.pdf')).is_file()
    checks['figures/'+n+'.png'] = (FIG/(n+'.png')).is_file()
nsamp = len(list(RES.glob('samples_*.npz')))
checks['samples_*.npz >= 32'] = nsamp >= 32

for k, v in checks.items():
    print(('[PASS] ' if v else '[FAIL] ') + k)
print('\nTOTAL: %d/%d PASS' % (sum(checks.values()), len(checks)))
assert all(checks.values()), 'some artifacts are missing'

---

**完成。** 如需真正从头复算全部数值，把最上方 `MODE` 改为 `'full'` 再运行全部（约 30 分钟，取决于算力）。